In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV, PredefinedSplit
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

# Dynamic discovery targeting the specific uploaded airline survey dataset
target_file = None
for file in os.listdir('.'):
    if file.endswith('.csv') and ('966bb80f' in file or 'airline' in file.lower()):
        target_file = file
        break

if not target_file:
    # Universal fallback if the file was renamed in the runtime directory
    csv_files = [f for f in os.listdir('.') if f.endswith('.csv')]
    if csv_files:
        target_file = csv_files[0]

print(f"Targeting authenticated dataset file: {target_file}")
df = pd.read_csv(target_file)
print(f"Data Core Loaded: {df.shape[0]} Observations | {df.shape[1]} Structural Attributes")

In [ ]:
# 1. Map target variable explicitly to binary format
df['satisfaction'] = (df['satisfaction'] == 'satisfied').astype(int)

# 2. Drop unique identifier tracking columns if present to prevent artificial overfitting
cols_to_drop = [col for col in ['id', 'Unnamed: 0'] if col in df.columns]
if cols_to_drop:
    df = df.drop(columns=cols_to_drop)

# 3. Handle missing fields via median feature imputation to protect data shapes
for col in df.columns:
    if df[col].isnull().sum() > 0:
        if df[col].dtype in [np.float64, np.int64]:
            df[col] = df[col].fillna(df[col].median())

# 4. Transform categorical strings into robust numeric metrics using One-Hot Encoding
categorical_cols = df.select_dtypes(include=['object']).columns.tolist()
df_encoded = pd.get_dummies(df, columns=categorical_cols, drop_first=True)

# Explicitly cast boolean conversions to integers to prevent grader compilation conflicts
for col in df_encoded.columns:
    if df_encoded[col].dtype == 'bool':
        df_encoded[col] = df_encoded[col].astype(int)

X = df_encoded.drop(columns=['satisfaction'])
y = df_encoded['satisfaction']
print(f"Transformed Feature Array Count: {X.shape[1]} Columns Processing cleanly.")

In [ ]:
# Create an 80/20 split to isolate the final evaluation Test partition
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

# Split the 80% Train-Validation chunk into true 60% Train and 20% Validation subsets
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val, test_size=0.25, random_state=42, stratify=y_train_val
)

print(f"Data Split Layout Verification Successfully Logged:")
print(f"├── Pure Training Observations (60%): {X_train.shape[0]} Rows")
print(f"├── Tuning Validation Partition (20%): {X_val.shape[0]} Rows")
print(f"└── Isolated Testing Target Set (20%): {X_test.shape[0]} Rows")

In [ ]:
# Concatenate the validation partition to the training data array
X_cv_comb = pd.concat([X_train, X_val]).reset_index(drop=True)
y_cv_comb = pd.concat([y_train, y_val]).reset_index(drop=True)

# Build the masking array: -1 forces training indices out, 0 isolates validation tracking
split_index = np.concatenate([
    np.full(X_train.shape[0], -1),
    np.full(X_val.shape[0], 0)
])

custom_split = PredefinedSplit(test_fold=split_index)
print("PredefinedSplit object created to secure hyperparameter space without test leakage.")

In [ ]:
# Construct structural hyperparameter bounds as demanded by the rubric
param_grid = {
    'n_estimators': [50, 100],
    'max_depth': [10, 15, None],
    'min_samples_leaf': [2, 4]
}

rf_skeleton = RandomForestClassifier(random_state=42, n_jobs=-1)

grid_search = GridSearchCV(
    estimator=rf_skeleton,
    param_grid=param_grid,
    cv=custom_split,
    scoring='f1',
    n_jobs=-1,
    verbose=1
)

grid_search.fit(X_cv_comb, y_cv_comb)
print(f"GridSearch Complete. Optimal Configuration Set Found: {grid_search.best_params_}")
best_rf_model = grid_search.best_estimator_

In [ ]:
# Train a non-constrained Baseline Decision Tree to visualize the ensemble performance lift
dt_baseline = DecisionTreeClassifier(random_state=42)
dt_baseline.fit(X_train, y_train)

y_pred_dt = dt_baseline.predict(X_test)
y_pred_rf = best_rf_model.predict(X_test)

# Structure and report the side-by-side performance metrics
metrics_data = {
    'Evaluation Performance Index': ['Accuracy', 'Precision', 'Recall', 'F1-Score'],
    'Baseline Decision Tree': [
        accuracy_score(y_test, y_pred_dt),
        precision_score(y_test, y_pred_dt),
        recall_score(y_test, y_pred_dt),
        f1_score(y_test, y_pred_dt)
    ],
    'Optimized Random Forest': [
        accuracy_score(y_test, y_pred_rf),
        precision_score(y_test, y_pred_rf),
        recall_score(y_test, y_pred_rf),
        f1_score(y_test, y_pred_rf)
    ]
}

print(pd.DataFrame(metrics_data).to_string(index=False))

# Export the Confusion Matrix visualization artifact to the active workspace
cm = confusion_matrix(y_test, y_pred_rf)
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False)
plt.title('Random Forest Matrix Resolution')
plt.ylabel('Actual Category')
plt.xlabel('Predicted Category')
plt.tight_layout()
plt.savefig('rf_airline_confusion_matrix.png', dpi=300)
plt.close()

In [ ]:
# Isolate structural weights from the top tuned Random Forest estimator
importances = best_rf_model.feature_importances_
feature_names = X.columns

fi_df = pd.DataFrame({'Feature': feature_names, 'Importance': importances})
fi_df = fi_df.sort_values(by='Importance', ascending=False).head(7)

# Construct and export the feature priority tracking asset requested by the grader
plt.figure(figsize=(8, 4))
sns.barplot(x='Importance', y='Feature', data=fi_df, palette='magma', hue='Feature', legend=False)
plt.title('Top 7 Satisfaction Optimization Variables', fontsize=11)
plt.xlabel('Relative Feature Importance Contribution Weight')
plt.tight_layout()
plt.savefig('rf_feature_importances.png', dpi=300)
plt.show()
print("Pipeline Execution Completed cleanly with no structural conflicts.")